# Datasets

Generally, we have several QUBO problems to solve — possibly with known solutions — forming a
dataset. `qubosolver` provides ways to generate, save, and load such datasets.

A `Dataset` is a flat, list-like collection of QUBO instances, each optionally paired with a
`Solution`.

## Creating Datasets

Here is how to generate a dataset of random QUBO instances and save it to disk.

In [ ]:
import torch
from pathlib import Path
from qubosolver import Dataset, torch_rng
 
dataset = Dataset.from_random(
    n_matrices=3,
    matrix_dim=5,
    densities=[0.6],
    coefficient_bounds=(-100.0, 100.0),
    rng=torch_rng(154),
    negative_offdiag_rate=0.0,
)

output_directory = Path.cwd() / "qubosolver_logs" / "tutorial" / "01-dataset-generation-and-loading" 
output_directory.mkdir(parents=True, exist_ok=True)
file_path = output_directory / "dataset.bin"

with file_path.open("wb") as f:
    dataset.save(f)
print(f"Dataset saved to {file_path}")

Dataset saved to /home/bussy/work/qubo-solver.git/251-simplify-api-further/docs/tutorial/qubosolver_logs/tutorial/01-dataset-generation-and-loading/dataset.bin


## Load datasets

Here we load the dataset back from disk and show it can be used the same way.

In [ ]:
with file_path.open("rb") as f:
    dataset = Dataset.load(f)
print(f"Dataset loaded from {file_path}")

Dataset loaded from /home/bussy/work/qubo-solver.git/251-simplify-api-further/docs/tutorial/qubosolver_logs/tutorial/01-dataset-generation-and-loading/dataset.bin


A `Dataset` behaves like a flat, index-based sequence of `(Instance, Solution)` pairs:
 - `dataset[i]` returns the `i`-th instance together with its solution.
 - if no solutions were provided when the dataset was built (as is the case here), each entry
   comes back with an empty `Solution`.

### Accessing a single entry by index

In [ ]:
from qubosolver import analysis

instance, solution = dataset[1]
print(f"Instance:\n{instance}\n")
print(f"Matrix:\n{instance.matrix}\n")
print(f"Solution:\n{analysis.to_dataframe([solution])}")

Instance:
'Instance of size = 5, density = 0.6'

Matrix:
tensor([[ 25.2829,   3.9080, 100.0000,  42.9466,  75.9399],
        [  3.9080,  78.8856,   0.0000,  82.5110,  84.9835],
        [100.0000,   0.0000, -87.7800,   0.0000,   0.0000],
        [ 42.9466,  82.5110,   0.0000,   0.0000,   0.0000],
        [ 75.9399,  84.9835,   0.0000,   0.0000,  -0.0000]],
       dtype=torch.float64)

Solution:
Empty DataFrame
Columns: [labels, bitstrings, costs, counts, probs]
Index: []


### Iterate over all instances

In [ ]:
for instance, solution in dataset:
    print("---------------------------")
    print(f"Instance:\n{instance}\n")
    print(f"Matrix:\n{instance.matrix}\n")
    print(f"Solution:\n{analysis.to_dataframe([solution])}")

---------------------------
Instance:
'Instance of size = 5, density = 0.6'

Matrix:
tensor([[ 52.4736,  48.5397,   0.0000,  87.4712, 100.0000],
        [ 48.5397,  -0.0000,   0.0000,  96.5363,   0.0000],
        [  0.0000,   0.0000,  83.7143,  32.5826,  69.1467],
        [ 87.4712,  96.5363,  32.5826, -77.0557,   0.0000],
        [100.0000,   0.0000,  69.1467,   0.0000,  -0.0000]],
       dtype=torch.float64)

Solution:
Empty DataFrame
Columns: [labels, bitstrings, costs, counts, probs]
Index: []
---------------------------
Instance:
'Instance of size = 5, density = 0.6'

Matrix:
tensor([[ 25.2829,   3.9080, 100.0000,  42.9466,  75.9399],
        [  3.9080,  78.8856,   0.0000,  82.5110,  84.9835],
        [100.0000,   0.0000, -87.7800,   0.0000,   0.0000],
        [ 42.9466,  82.5110,   0.0000,   0.0000,   0.0000],
        [ 75.9399,  84.9835,   0.0000,   0.0000,  -0.0000]],
       dtype=torch.float64)

Solution:
Empty DataFrame
Columns: [labels, bitstrings, costs, counts, probs]
Inde

### Attaching solutions

Solutions are computed independently (here with an exact brute-force solver) and collected into
a list. A new `Dataset` can then be built from the same matrices plus that list of `Solution`
objects.

In [ ]:
from qubosolver import analysis, solvers

solutions = []

for instance, _ in dataset:
    solution = solvers.brute_force(instance, max_bitstrings=1)
    solutions.append(solution)

matrices = torch.stack([Q.matrix for Q, _ in dataset], dim=-1)
dataset_with_solutions = Dataset(matrices=matrices, solutions=solutions)

for instance, solution in dataset_with_solutions:
    print("---------------------------")
    print(f"Instance:\n{instance}\n")
    print(f"Matrix:\n{instance.matrix}\n")
    print(f"Solution:\n{analysis.to_dataframe([solution])}")

---------------------------
Instance:
'Instance of size = 5, density = 0.6'

Matrix:
tensor([[ 52.4736,  48.5397,   0.0000,  87.4712, 100.0000],
        [ 48.5397,  -0.0000,   0.0000,  96.5363,   0.0000],
        [  0.0000,   0.0000,  83.7143,  32.5826,  69.1467],
        [ 87.4712,  96.5363,  32.5826, -77.0557,   0.0000],
        [100.0000,   0.0000,  69.1467,   0.0000,  -0.0000]],
       dtype=torch.float64)

Solution:
  labels bitstrings      costs  counts  probs
0      0      00010 -77.055653     1.0    1.0
---------------------------
Instance:
'Instance of size = 5, density = 0.6'

Matrix:
tensor([[ 25.2829,   3.9080, 100.0000,  42.9466,  75.9399],
        [  3.9080,  78.8856,   0.0000,  82.5110,  84.9835],
        [100.0000,   0.0000, -87.7800,   0.0000,   0.0000],
        [ 42.9466,  82.5110,   0.0000,   0.0000,   0.0000],
        [ 75.9399,  84.9835,   0.0000,   0.0000,  -0.0000]],
       dtype=torch.float64)

Solution:
  labels bitstrings      costs  counts  probs
0      0    